<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller: Un Robot de Bodega que Aprende a Decidir 🤖📦
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 06
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/06_Aprendizaje_por_Refuerzo_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 06 (Aprendizaje por Refuerzo)** con un entorno nuevo, construido por ti en **NumPy puro**: un **robot de bodega** que debe llegar a la estación de carga evitando derrames, sobre un piso resbaladizo. Al terminar serás capaz de:

1. **Calcular retornos descontados** (suma directa y recursión hacia atrás) y estimarlos por **Monte Carlo**.
2. **Evaluar una política de forma exacta** (sistema lineal de Bellman) y hallar la **política óptima por iteración de valor**, verificándola con propiedades de optimalidad.
3. **Entrenar un agente con Q-learning tabular** a partir de la interacción y juzgarlo por el **valor exacto de su política voraz**, no por una cifra que dependa de la semilla.
4. **Programar el gradiente de política (REINFORCE)** con *baseline* en un bandido de tres brazos, verificando el gradiente del *softmax* con diferencias finitas.

> 📍 **Antes de empezar:** repasa [Fundamentos de RL y MDPs](../06%20-%20Aprendizaje%20por%20Refuerzo/00_Fundamentos_RL_y_MDPs.ipynb), [Funciones de valor y Q-learning](../06%20-%20Aprendizaje%20por%20Refuerzo/01_Funciones_de_Valor_y_Q_Learning.ipynb) y [Gradientes de política](../06%20-%20Aprendizaje%20por%20Refuerzo/02_Gradientes_de_Politica.ipynb). Si necesitas más guía, existe la [versión Para Dummies](Para%20Dummies/06_Aprendizaje_por_Refuerzo_Hands_On_Dummies.ipynb) de este taller.

---
## Instrucciones

1. **Resuelve de forma autónoma:** completa cada celda de código marcada con `# TODO:` respetando los **nombres de variables y funciones** que se piden (las autoverificaciones los usan).
2. Después de cada reto hay una celda de **autoverificación** con `assert`. Mientras el reto no esté resuelto **fallará con un mensaje** que te orienta; cuando todo esté bien imprimirá un ✅.
3. No borres ni modifiques las celdas de autoverificación ni la celda de preparación. **No se usa ninguna biblioteca de RL** (ni Gymnasium ni PyTorch): solo NumPy.
4. Las autoverificaciones comprueban **propiedades** (valores exactos conocidos, optimalidad, convergencia con tolerancia amplia), no cifras que dependan de la semilla.
5. Antes de entregar ejecuta **Kernel → Restart & Run All** y comprueba que no queda ningún error ni ningún `# TODO` sin resolver. El taller corre en menos de un minuto.

---
## Preparación del entorno ⚙️

Ejecuta la celda siguiente (**no la modifiques**). Construye el **robot de bodega** como un MDP finito con tensores explícitos:

- **Mapa** (4×5): `S` inicio, `G` estación de carga (**+20**, terminal), `X` derrame (**−10**, terminal), `#` estantería (no se puede entrar: el robot rebota y se queda), `.` pasillo. **Cada paso cuesta −1**.
- **Piso resbaladizo:** la acción elegida ocurre con probabilidad 0,8 y con 0,1 cada una de las dos direcciones perpendiculares. Acciones: `0` ←, `1` ↓, `2` →, `3` ↑. Descuento $\gamma=0{,}95$.
- `mdp.P[s, a, s']`, `mdp.R[s, a, s']`, `mdp.r[s, a]` $=\mathbb E[R\mid s,a]$, `mdp.terminales`, `mdp.s0`, `mdp.gamma` y `mdp.step(s, a, rng)` $\to (s', r, \text{terminado})$. Los estados terminales y las estanterías son absorbentes y no dan recompensa, por lo que su valor es 0.
- `simular_episodio(mdp, pi, rng)`: juega un episodio con la política estocástica `pi` (matriz `nS × nA`) y devuelve la lista de recompensas.
- `pi_aleatoria`: la política uniforme. `mostrar_politica(pol)` dibuja una política determinista sobre el mapa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 7

MAPA = ["S....",
        ".#X..",
        "...#.",
        "X...G"]
MOV = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}      # 0 izquierda, 1 abajo, 2 derecha, 3 arriba


def listo(*valores):
    """True si ninguna variable sigue sin resolver (None o ...)."""
    return all(v is not None and v is not Ellipsis for v in valores)


class MDP:
    """MDP finito con estados terminales y estanterias absorbentes."""

    def __init__(self, mapa, gamma=0.95, deslizamiento=0.2):
        self.mapa, self.gamma = mapa, gamma
        self.nf, self.nc = len(mapa), len(mapa[0])
        self.nS, self.nA = self.nf * self.nc, 4
        P = np.zeros((self.nS, 4, self.nS)); R = np.zeros((self.nS, 4, self.nS))
        term = np.zeros(self.nS, dtype=bool)
        for f in range(self.nf):
            for c in range(self.nc):
                s, ch = f * self.nc + c, mapa[f][c]
                if ch == "S":
                    self.s0 = s
                if ch in "XG":
                    term[s] = True; P[s, :, s] = 1.0; continue
                if ch == "#":
                    P[s, :, s] = 1.0; continue
                for a in range(4):
                    for d, p in (((a - 1) % 4, deslizamiento / 2), (a, 1 - deslizamiento), ((a + 1) % 4, deslizamiento / 2)):
                        f2 = min(max(f + MOV[d][0], 0), self.nf - 1); c2 = min(max(c + MOV[d][1], 0), self.nc - 1)
                        if mapa[f2][c2] == "#":
                            f2, c2 = f, c
                        s2 = f2 * self.nc + c2
                        P[s, a, s2] += p
                        R[s, a, s2] = 20.0 if mapa[f2][c2] == "G" else (-10.0 if mapa[f2][c2] == "X" else -1.0)
        self.P, self.R, self.terminales = P, R, term
        self.r = (P * R).sum(axis=2)
        self.cdf = np.cumsum(P, axis=2); self.cdf[..., -1] = 1.0

    def step(self, s, a, rng):
        s2 = min(int(np.searchsorted(self.cdf[s, a], rng.random(), side="right")), self.nS - 1)
        return s2, float(self.R[s, a, s2]), bool(self.terminales[s2])


def simular_episodio(mdp, pi, rng, max_pasos=200):
    """Devuelve la lista de recompensas de un episodio jugado con la politica estocastica pi."""
    s, recompensas = mdp.s0, []
    for _ in range(max_pasos):
        a = int(rng.choice(mdp.nA, p=pi[s]))
        s, r, terminado = mdp.step(s, a, rng)
        recompensas.append(r)
        if terminado:
            break
    return recompensas


def mostrar_politica(pol):
    flechas = {0: "←", 1: "↓", 2: "→", 3: "↑"}
    for f in range(mdp.nf):
        print(" ".join(ch if ch in "#XG" else flechas[int(pol[f * mdp.nc + c])] for c, ch in enumerate(mdp.mapa[f])))


mdp = MDP(MAPA, gamma=0.95, deslizamiento=0.2)
pi_aleatoria = np.full((mdp.nS, mdp.nA), 1 / mdp.nA)
print("Estados:", mdp.nS, "| acciones:", mdp.nA, "| estado inicial:", mdp.s0, "| terminales:", np.flatnonzero(mdp.terminales))
assert np.allclose(mdp.P.sum(axis=2), 1), "El tensor P debe sumar 1 sobre s'"

---
## Reto 1: Retornos y estimación Monte Carlo 🎯 (15 puntos)

**Contexto.** Antes de optimizar nada, necesitas medir cuánto vale un episodio. El **retorno descontado** de una secuencia de recompensas $r_1,r_2,\dots,r_T$ es

$$
G_0=\sum_{k=0}^{T-1}\gamma^{k}\,r_{k+1},\qquad G_t=r_{t+1}+\gamma\,G_{t+1}.
$$

**Enunciado**

1. Implementa `retorno(recompensas, gamma)` (suma directa) y `retornos_hacia_atras(recompensas, gamma)`, que devuelva un arreglo NumPy con **todos** los $G_t$ aplicando la recursión desde el final.
2. Con `rng = np.random.default_rng(SEED)`, simula 1000 episodios de la política `pi_aleatoria` con `simular_episodio` y guarda en `retorno_medio_aleatoria` el promedio de sus retornos descontados con `mdp.gamma`.

**Criterios de aceptación**

- `retorno([-1, -1, -1, 10], 0.9)` $=4{,}58$; con $\gamma=1$ es la suma simple; con $\gamma=0$ es la primera recompensa; sin recompensas es $0$.
- `retornos_hacia_atras` cumple $G_t=r_{t+1}+\gamma G_{t+1}$ en cada paso y $G_0$ coincide con `retorno`.
- La estimación Monte Carlo cae a menos de 1,0 del valor **exacto** de la política aleatoria en el estado inicial (calculado de forma independiente por la autoverificación).

In [ ]:
def retorno(recompensas, gamma):
    """G_0 = suma_k gamma^k * r_{k+1} (suma directa)."""
    # TODO
    pass


def retornos_hacia_atras(recompensas, gamma):
    """Arreglo con G_t para todo t, usando G_t = r_{t+1} + gamma * G_{t+1} desde el final."""
    # TODO
    pass


rng = np.random.default_rng(SEED)
retorno_medio_aleatoria = None   # TODO: promedio de 1000 retornos descontados de simular_episodio(mdp, pi_aleatoria, rng)

In [ ]:
# ---- Autoverificacion del Reto 1 (no modificar) ----
_r = retorno([-1, -1, -1, 10], 0.9)
assert _r is not None, "Reto 1: retorno todavia no devuelve ningun valor."
assert abs(_r - 4.58) < 1e-9, f"Reto 1: retorno([-1,-1,-1,10], 0.9) debe ser 4.58 (-1 - 0.9 - 0.81 + 7.29) y obtuviste {_r}."
assert abs(retorno([2, 3, 4], 1.0) - 9) < 1e-9, "Reto 1: con gamma = 1 el retorno es la suma simple."
assert abs(retorno([5, 100, 100], 0.0) - 5) < 1e-9, "Reto 1: con gamma = 0 solo cuenta la primera recompensa."
assert retorno([], 0.9) == 0, "Reto 1: un episodio sin recompensas tiene retorno 0."
_rec = [0, 0, -1, 0, 5, 0, 10]
_G = retornos_hacia_atras(_rec, 0.9)
assert _G is not None and len(_G) == len(_rec), "Reto 1: retornos_hacia_atras debe devolver un G_t por cada recompensa."
assert abs(_G[-1] - _rec[-1]) < 1e-9, "Reto 1: el ultimo G_t es la ultima recompensa."
assert all(abs(_G[t] - (_rec[t] + 0.9 * _G[t + 1])) < 1e-9 for t in range(len(_rec) - 1)), "Reto 1: debe cumplirse G_t = r_{t+1} + gamma * G_{t+1}."
assert abs(_G[0] - retorno(_rec, 0.9)) < 1e-9, "Reto 1: G_0 de la recursion debe coincidir con retorno()."
assert listo(retorno_medio_aleatoria), "Reto 1: falta retorno_medio_aleatoria (paso 2)."
_Ppi = np.einsum("sa,sat->st", pi_aleatoria, mdp.P); _rpi = (pi_aleatoria * mdp.r).sum(axis=1)
_V = np.linalg.solve(np.eye(mdp.nS) - mdp.gamma * _Ppi, _rpi)
assert abs(retorno_medio_aleatoria - _V[mdp.s0]) < 1.0, f"Reto 1: tu estimacion Monte Carlo ({retorno_medio_aleatoria:.2f}) deberia estar a menos de 1.0 del valor exacto ({_V[mdp.s0]:.2f}). Usa 1000 episodios, mdp.gamma y pi_aleatoria."
print(f"✅ Reto 1 correcto | Monte Carlo = {retorno_medio_aleatoria:.2f} vs exacto = {_V[mdp.s0]:.2f} (la politica aleatoria es mala)")

---
## Reto 2: Evaluación exacta e iteración de valor 🧮 (20 puntos)

**Contexto.** Como conoces $P$ y $R$, no necesitas simular: puedes **resolver** el problema. Para una política $\pi$ la ecuación de Bellman de expectativa es lineal,

$$
v_\pi=r_\pi+\gamma P_\pi v_\pi\ \Longrightarrow\ (I-\gamma P_\pi)\,v_\pi=r_\pi,
$$

con $P_\pi(s,s')=\sum_a\pi(a\mid s)P(s,a,s')$ y $r_\pi(s)=\sum_a\pi(a\mid s)\,r(s,a)$. La **iteración de valor** aplica una y otra vez el operador de optimalidad $v\leftarrow\max_a\big[r(s,a)+\gamma\sum_{s'}P(s,a,s')v(s')\big]$ hasta converger.

**Enunciado**

1. Implementa `evaluar_politica(mdp, pi)` con `np.linalg.solve` (`pi` es una matriz `nS × nA`) y devuelve $v_\pi$.
2. Implementa `iteracion_de_valor(mdp, tol=1e-10)` que devuelva la tupla `(V, Q, politica)`: $v_\ast$, $q_\ast(s,a)=r(s,a)+\gamma\sum_{s'}P\,v_\ast$ y la política voraz `politica = Q.argmax(axis=1)`.
3. Calcula `V_aleatoria = evaluar_politica(mdp, pi_aleatoria)` y `V_star, Q_star, pol_star = iteracion_de_valor(mdp)`.

**Criterios de aceptación**

- `V_aleatoria` cumple la ecuación de Bellman de expectativa y `V_star` la de **optimalidad** (residuo $<10^{-6}$).
- $v_\ast\ge v_\pi$ en **todos** los estados y $v_\ast=0$ en los terminales.
- Evaluar `pol_star` (como política determinista) devuelve $v_\ast$: la política voraz es óptima.
- En las dos casillas contiguas a la estación de carga la política óptima va **hacia la meta**.

In [ ]:
def evaluar_politica(mdp, pi):
    """Resuelve (I - gamma * P_pi) v = r_pi y devuelve v_pi (arreglo de tamano nS)."""
    # TODO: P_pi = np.einsum("sa,sat->st", pi, mdp.P);  r_pi = (pi * mdp.r).sum(axis=1)
    pass


def iteracion_de_valor(mdp, tol=1e-10):
    """Devuelve (V, Q, politica) con V = v*, Q = q* y politica = argmax_a Q."""
    # TODO: V = ceros; repite Q = mdp.r + mdp.gamma * (mdp.P @ V); V_nuevo = Q.max(axis=1)
    #       hasta que max |V_nuevo - V| < tol.  Al final recalcula Q con la V final.
    pass


V_aleatoria = None
V_star, Q_star, pol_star = None, None, None   # TODO: usa tus dos funciones

In [ ]:
# ---- Autoverificacion del Reto 2 (no modificar) ----
assert listo(V_aleatoria, V_star, Q_star, pol_star), "Reto 2: faltan V_aleatoria, V_star, Q_star o pol_star (paso 3). Revisa que tus funciones devuelvan valores."
V_aleatoria, V_star, Q_star, pol_star = (np.asarray(x) for x in (V_aleatoria, V_star, Q_star, pol_star))
assert V_aleatoria.shape == (mdp.nS,) and V_star.shape == (mdp.nS,), "Reto 2: V_aleatoria y V_star deben tener un valor por estado."
assert Q_star.shape == (mdp.nS, mdp.nA) and pol_star.shape == (mdp.nS,), "Reto 2: Q_star debe ser (nS, nA) y pol_star de tamano nS."
_Ppi = np.einsum("sa,sat->st", pi_aleatoria, mdp.P); _rpi = (pi_aleatoria * mdp.r).sum(axis=1)
assert np.abs(V_aleatoria - (_rpi + mdp.gamma * _Ppi @ V_aleatoria)).max() < 1e-6, "Reto 2: V_aleatoria no cumple la ecuacion de Bellman de expectativa. Resuelve (I - gamma P_pi) v = r_pi."
_Qchk = mdp.r + mdp.gamma * (mdp.P @ V_star)
assert np.abs(V_star - _Qchk.max(axis=1)).max() < 1e-6, "Reto 2: V_star no cumple la ecuacion de optimalidad de Bellman; itera hasta converger (tol pequena)."
assert np.abs(Q_star - _Qchk).max() < 1e-6, "Reto 2: Q_star debe ser r + gamma * P @ V_star con la V final."
assert np.all(V_star >= V_aleatoria - 1e-8), "Reto 2: v* nunca puede ser menor que v_pi en ningun estado."
assert V_star[mdp.s0] > V_aleatoria[mdp.s0] + 10, "Reto 2: la politica optima deberia superar por mucho a la aleatoria en el estado inicial."
assert np.allclose(V_star[mdp.terminales], 0, atol=1e-8), "Reto 2: el valor de los estados terminales debe ser 0."
assert np.array_equal(pol_star, Q_star.argmax(axis=1)), "Reto 2: pol_star debe ser Q_star.argmax(axis=1)."
_pi_star = np.eye(mdp.nA)[pol_star]
_V_eval = np.linalg.solve(np.eye(mdp.nS) - mdp.gamma * np.einsum("sa,sat->st", _pi_star, mdp.P), (_pi_star * mdp.r).sum(axis=1))
assert np.abs(_V_eval - V_star).max() < 1e-6, "Reto 2: evaluar pol_star deberia devolver v*; la politica voraz no es optima."
_izq_G, _arriba_G = 3 * mdp.nc + 3, 2 * mdp.nc + 4        # casilla a la izquierda de G y casilla encima de G
assert pol_star[_izq_G] == 2, "Reto 2: junto a la estacion de carga (a su izquierda) la accion optima es ir a la derecha (2)."
assert pol_star[_arriba_G] == 1, "Reto 2: encima de la estacion de carga la accion optima es ir hacia abajo (1)."
print(f"✅ Reto 2 correcto | v*(inicio) = {V_star[mdp.s0]:.2f} vs politica aleatoria = {V_aleatoria[mdp.s0]:.2f}")
mostrar_politica(pol_star)

---
## Reto 3: Q-learning tabular a partir de la interacción 🤖 (25 puntos)

**Contexto.** En la práctica el robot **no conoce** $P$ ni $R$: solo puede actuar y observar $(s,a,r,s')$ mediante `mdp.step`. *Q-learning* aprende $q_\ast$ con la actualización *off-policy*

$$
Q(s,a)\leftarrow Q(s,a)+\alpha\Big[r+\gamma\,(1-\text{terminado})\max_{a'}Q(s',a')-Q(s,a)\Big].
$$

**Enunciado**

1. Implementa `q_learning(mdp, episodios, alpha, epsilon, semilla, max_pasos=200)` que devuelva `(Q, retornos)`: la tabla `Q` (`nS × nA`, inicializada en ceros) y la lista con la **suma de recompensas** de cada episodio. Usa política $\varepsilon$-*greedy* (con probabilidad `epsilon` una acción al azar; si no, `argmax` de `Q[s]`) y un `np.random.default_rng(semilla)` propio. Un estado terminal **no** se actualiza ni tiene valor futuro.
2. Entrénalo con `EPISODIOS = 4000`, `ALPHA = 0.1`, `EPSILON = 0.2` y `semilla=0`; guarda `Q_ql, ret_ql` y la política voraz `pol_ql = Q_ql.argmax(axis=1)`.

**Criterios de aceptación** (con tolerancia amplia, no dependen de una semilla concreta; la autoverificación también reentrena con otras dos semillas)

- `Q_ql` tiene la forma correcta y vale 0 en los terminales.
- El **valor exacto** de la política voraz `pol_ql` en el estado inicial queda a menos de 2,0 de $v_\ast$ y supera por más de 10 a la política aleatoria.
- Las casillas contiguas a la meta tienen la acción óptima, y el retorno medio del 10 % final de los episodios supera al del 10 % inicial.

In [ ]:
def q_learning(mdp, episodios, alpha, epsilon, semilla, max_pasos=200):
    """Q-learning tabular. Devuelve (Q, retornos) con la suma de recompensas de cada episodio."""
    # TODO: rng = np.random.default_rng(semilla);  Q = ceros (nS, nA)
    #   por episodio: s = mdp.s0 y, hasta max_pasos o terminar:
    #     a = accion aleatoria con prob. epsilon; si no, argmax de Q[s] (rompe empates al azar o con ruido minimo)
    #     s2, r, terminado = mdp.step(s, a, rng)
    #     objetivo = r + gamma * max(Q[s2]) * (1 - terminado);  Q[s, a] += alpha * (objetivo - Q[s, a])
    pass


EPISODIOS, ALPHA, EPSILON = 4000, 0.1, 0.2
Q_ql, ret_ql = None, None    # TODO: llama a q_learning(mdp, EPISODIOS, ALPHA, EPSILON, semilla=0)
pol_ql = None                # TODO: politica voraz de Q_ql

In [ ]:
# ---- Autoverificacion del Reto 3 (no modificar) ----
assert listo(Q_ql, ret_ql, pol_ql), "Reto 3: faltan Q_ql, ret_ql o pol_ql (paso 2). Revisa que q_learning devuelva (Q, retornos)."
assert np.asarray(Q_ql).shape == (mdp.nS, mdp.nA), "Reto 3: Q debe tener forma (nS, nA)."
assert len(ret_ql) == EPISODIOS, "Reto 3: retornos debe tener un valor por episodio."
assert np.abs(np.asarray(Q_ql)[mdp.terminales]).max() == 0, "Reto 3: los estados terminales no se actualizan: su fila de Q debe quedarse en 0."
assert np.array_equal(np.asarray(pol_ql), np.asarray(Q_ql).argmax(axis=1)), "Reto 3: pol_ql debe ser Q_ql.argmax(axis=1)."


def _valor_exacto(pol):
    pi = np.eye(mdp.nA)[np.asarray(pol)]
    return np.linalg.solve(np.eye(mdp.nS) - mdp.gamma * np.einsum("sa,sat->st", pi, mdp.P), (pi * mdp.r).sum(axis=1))


_Vs = np.zeros(mdp.nS)                                    # v* calculado de forma independiente (iteracion de valor)
for _ in range(2000):
    _Vs = (mdp.r + mdp.gamma * (mdp.P @ _Vs)).max(axis=1)
_Va = np.linalg.solve(np.eye(mdp.nS) - mdp.gamma * np.einsum("sa,sat->st", pi_aleatoria, mdp.P), (pi_aleatoria * mdp.r).sum(axis=1))
for _nombre, _Q, _ret in [("semilla 0 (la tuya)", np.asarray(Q_ql), ret_ql)] + [(f"semilla {s}", *q_learning(mdp, EPISODIOS, ALPHA, EPSILON, semilla=s)) for s in (1, 2)]:
    _pol = np.asarray(_Q).argmax(axis=1); _Vg = _valor_exacto(_pol)
    assert _Vg[mdp.s0] >= _Vs[mdp.s0] - 2.0, f"Reto 3 ({_nombre}): el valor exacto de tu politica voraz en el inicio es {_Vg[mdp.s0]:.2f}, a mas de 2.0 de v* = {_Vs[mdp.s0]:.2f}. Revisa la actualizacion (objetivo, terminales, epsilon)."
    assert _Vg[mdp.s0] > _Va[mdp.s0] + 10, f"Reto 3 ({_nombre}): tu politica deberia superar por mas de 10 a la aleatoria ({_Va[mdp.s0]:.2f})."
    assert _pol[3 * mdp.nc + 3] == 2 and _pol[2 * mdp.nc + 4] == 1, f"Reto 3 ({_nombre}): junto a la meta la politica deberia ir hacia ella (derecha a la izquierda de G, abajo encima de G)."
    _n = len(_ret) // 10
    assert np.mean(_ret[-_n:]) > np.mean(_ret[:_n]), f"Reto 3 ({_nombre}): el retorno medio del 10 % final de los episodios deberia superar al del 10 % inicial (hay aprendizaje)."
assert abs(np.asarray(Q_ql)[mdp.s0].max() - _Vs[mdp.s0]) < 2.5, "Reto 3: max_a Q[s0, a] deberia estar a menos de 2.5 de v*(inicio)."
print(f"✅ Reto 3 correcto | valor exacto de tu politica en el inicio = {_valor_exacto(pol_ql)[mdp.s0]:.2f} (v* = {_Vs[mdp.s0]:.2f})")
mostrar_politica(pol_ql)

---
## Reto 4: Gradiente de política (REINFORCE) en un bandido de tres brazos 🎰 (25 puntos)

**Contexto.** Hasta ahora aprendiste **valores**; ahora aprenderás la **política directamente**. El bandido tiene 3 brazos con recompensas gaussianas de medias `MU = [0.2, 0.5, 1.0]` y desviación `SIGMA = 0.5` (ya definidos). Parametrizas la política con un vector $\theta\in\mathbb R^3$ y $\pi_\theta(a)=\operatorname{softmax}(\theta)_a$.

Para el *softmax* se cumple $\nabla_\theta\log\pi_\theta(a)=e_a-\pi_\theta$ (con $e_a$ el vector canónico), y REINFORCE con *baseline* actualiza en cada paso

$$
\theta\leftarrow\theta+\alpha\,(r-b)\,\nabla_\theta\log\pi_\theta(a),
$$

con $b$ el promedio de las recompensas vistas hasta ahora (una *baseline* que no depende de la acción no introduce sesgo).

**Enunciado**

1. Implementa `softmax(z)` **numéricamente estable** (resta el máximo) y `grad_log_pi(theta, a)`.
2. Implementa `reinforce_bandido(pasos, lr, semilla)`: parte de $\theta=0$, en cada paso `n` (desde 1) muestrea la acción de $\pi_\theta$, obtiene `r = rng.normal(MU[a], SIGMA)`, actualiza $\theta$ con la *baseline* $b$ y **después** actualiza $b\leftarrow b+(r-b)/n$. Devuelve el $\theta$ final.
3. Guarda `pi_final = softmax(reinforce_bandido(3000, 0.05, 0))`.

**Criterios de aceptación**

- `softmax` suma 1, es estable con logits enormes (`[1000, 1001, 1002]`) y `grad_log_pi` coincide con **diferencias finitas** (error $<10^{-5}$).
- La política aprendida concentra más de 0,8 de probabilidad en el **mejor brazo** (índice 2), para la semilla 0 y para otras dos semillas que reentrena la autoverificación.

In [ ]:
MU = np.array([0.2, 0.5, 1.0])
SIGMA = 0.5


def softmax(z):
    """Softmax estable: resta el maximo antes de exponenciar."""
    # TODO
    pass


def grad_log_pi(theta, a):
    """Gradiente de log pi_theta(a) respecto a theta para la politica softmax."""
    # TODO: e_a - softmax(theta)
    pass


def reinforce_bandido(pasos, lr, semilla):
    """REINFORCE con baseline (promedio corriente de recompensas). Devuelve theta final."""
    # TODO: rng = np.random.default_rng(semilla); theta = ceros(3); b = 0.0
    #   para n = 1..pasos: p = softmax(theta); a = rng.choice(3, p=p); r = rng.normal(MU[a], SIGMA)
    #   theta += lr * (r - b) * grad_log_pi(theta, a);   luego  b += (r - b) / n
    pass


pi_final = None   # TODO: softmax(reinforce_bandido(3000, 0.05, 0))

In [ ]:
# ---- Autoverificacion del Reto 4 (no modificar) ----
_s = softmax(np.array([1.0, 2.0, 3.0]))
assert _s is not None and abs(np.sum(_s) - 1) < 1e-9 and np.all(_s > 0), "Reto 4: softmax debe devolver probabilidades positivas que sumen 1."
_big = softmax(np.array([1000.0, 1001.0, 1002.0]))
assert np.all(np.isfinite(_big)) and abs(_big.sum() - 1) < 1e-9, "Reto 4: softmax debe ser estable con logits enormes: resta el maximo antes de exponenciar."
assert abs(_big[2] - _s[2]) < 1e-9, "Reto 4: softmax no cambia al sumar una constante a todos los logits."
_rg = np.random.default_rng(1); _th = _rg.normal(size=3)
for _a in range(3):
    _g = grad_log_pi(_th, _a)
    assert _g is not None and np.asarray(_g).shape == (3,), "Reto 4: grad_log_pi debe devolver un vector de 3 componentes."
    assert abs(np.sum(_g)) < 1e-9, "Reto 4: las componentes de grad log pi suman 0 (e_a y pi suman 1 cada uno)."
    _num = np.zeros(3)
    for _j in range(3):
        _d = np.zeros(3); _d[_j] = 1e-6
        _num[_j] = (np.log(softmax(_th + _d)[_a]) - np.log(softmax(_th - _d)[_a])) / 2e-6
    assert np.abs(_g - _num).max() < 1e-5, f"Reto 4: grad_log_pi(theta, {_a}) no coincide con las diferencias finitas. Recuerda: e_a - pi."
assert listo(pi_final), "Reto 4: falta pi_final (paso 3)."
for _nombre, _p in [("semilla 0 (la tuya)", np.asarray(pi_final))] + [(f"semilla {s}", softmax(reinforce_bandido(3000, 0.05, s))) for s in (1, 2)]:
    assert abs(_p.sum() - 1) < 1e-9, "Reto 4: pi_final debe ser una distribucion de probabilidad."
    assert _p[2] > 0.8, f"Reto 4 ({_nombre}): la probabilidad del mejor brazo (indice 2) deberia superar 0.8 y es {_p[2]:.2f}. Revisa el signo de la actualizacion y el orden: primero theta con b, despues b."
print(f"✅ Reto 4 correcto | politica aprendida (semilla 0) = {np.round(pi_final, 3)} | mejor brazo: {int(np.argmax(MU))}")

---
## Reto 5: Conclusiones 🧠 (15 puntos)

*Sin código.* Escribe entre **5 y 8 líneas** que interpreten **tus** resultados (cita `retorno_medio_aleatoria`, `V_aleatoria[mdp.s0]`, `V_star[mdp.s0]`, el valor exacto de `pol_ql` y `pi_final`). Debes responder, como mínimo:

1. ¿Por qué el retorno de la política aleatoria es tan negativo y cómo lo corrige la política óptima? ¿Qué papel juegan el costo por paso y $\gamma$?
2. ¿Qué diferencia práctica hay entre **resolver** el MDP (Reto 2) y **aprender** con Q-learning (Reto 3)? ¿Qué necesita saber cada método del entorno?
3. ¿Por qué evaluamos a Q-learning con el **valor exacto de su política voraz** y no con la curva de retornos de entrenamiento? ¿Qué papel cumple $\varepsilon$?
4. En el bandido, ¿qué ganas con la *baseline* y por qué no introduce sesgo? ¿Qué pasaría si $\mathrm{lr}$ fuera 10 veces mayor?

**Tus conclusiones (edita esta celda):**

> *(escribe aquí)*

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: retorno descontado, recursión hacia atrás y estimación Monte Carlo | 15 |
| Reto 2: evaluación exacta e iteración de valor con propiedades de optimalidad | 20 |
| Reto 3: Q-learning tabular juzgado por el valor exacto de su política | 25 |
| Reto 4: REINFORCE con *baseline* y gradiente verificado por diferencias finitas | 25 |
| Reto 5: conclusiones (5-8 líneas, con cifras propias y razonamiento correcto) | 15 |
| **Total** | **100** |

*Las autoverificaciones comprueban propiedades y tolerancias amplias; el docente puede revisar además la calidad del código y de la interpretación.*

### ✅ Checklist de entrega

- [ ] Ejecuté **Restart & Run All** y no hay errores ni celdas `# TODO` sin resolver.
- [ ] Las cuatro autoverificaciones imprimen ✅.
- [ ] Respondí las cuatro preguntas de las conclusiones con cifras de mis propios resultados.
- [ ] No modifiqué las celdas de preparación ni de autoverificación.
- [ ] Entregué el notebook con los resultados visibles (salidas guardadas).

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>